In [ ]:
%%capture
!pip install unsloth
# 同时获取最新的版本 Unsloth！
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
!pip install --upgrade transformers torch peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 79.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.0/411.0 kB 29.2 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.50.3
    Uninstalling transformers-4.50.3:
      Successfully uninstalled transformers-4.50.3
  Attempting uninstall: peft
    Found existing installation: peft 0.14.0
    Uninstalling peft-0.14.0:
      Successfully uninstalled peft-0.14.0


In [ ]:
!pip install unsloth

In [ ]:
# 导入 Unsloth 库中的 FastLanguageModel 类
import unsloth
from unsloth import FastLanguageModel
import torch

# 设置模型输入序列的最大长度，单位为 token。这个值限制了每次模型处理的文本长度
max_seq_length = 4096

# 设置模型的数据类型，如果为 None，通常会默认使用 float32
dtype = None

# 设置是否以 4-bit 精度加载模型。设置为 True 可以减少内存占用和计算量，但可能会降低精度
load_in_4bit = True

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Failed to patch Gemma3ForConditionalGeneration.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
## 使用本地环境
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HUGGINGFACE_TOKEN')
login(hf_token)

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Venassa/DeepSeek-R1-Distill-Qwen-7B-Children_Narrative_Extraction-Fine-tune_version2",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    token = hf_token,
)

==((====))==  Unsloth 2025.3.19: Fast Qwen2 patching. Transformers: 4.51.2.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.88G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.33G [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/231 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/6.78k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/472 [00:00<?, ?B/s]

In [ ]:
prompt_style = """###指令: 从儿童语言文本中提取叙事事件和事件关系。文本以<sen>分隔。

**事件定义:** (谓语；主语；宾语；时间状语；地点状语)  [缺失信息用“无”，多主语逗号分隔]

**事件关系:** [并列、动机因果、心理因果、物理因果、使能因果]

**输出格式:**
* 事件: (谓语；主语；宾语；时间状语；地点状语)  [多个事件用空格分隔]
* 关系: 事件1 事件2 关系类型

严格按格式输出。

###问题: {user_input}
###回答: """

In [ ]:
# 中文问答问题
question = """成千上万的蜜蜂都飞过来了。<sen>然后呢因为蜜蜂追着他。<sen>然后呢蜂巢掉下来了。"""

# 构造用户输入
user_input = question.strip()

# 根据提示模板和问题构造输入
inputs = tokenizer([prompt_style.format(user_input=user_input)], return_tensors="pt").to("cuda")

# 启动快速推理
FastLanguageModel.for_inference(model)  # Unsloth 已实现2倍加速推理！

# 模型生成答案
outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=1024,
    use_cache=True,
)

# 解码输出并提取回答内容
# response = tokenizer.batch_decode(outputs)
# print(response[0].split("回答:")[1].strip())

full_response = tokenizer.batch_decode(outputs)[0]
if "</think>" in full_response:
    final_answer = full_response.split("</think>")[-1].strip()  # 取最后一段
else:
    final_answer = full_response  # 容错处理
print(final_answer)

事件: (成千上万的蜜蜂；飞过来了；无；无；无) (因为蜜蜂追着他；无；无；无；无) (蜂巢掉下来了；无；无；无；无)  
关系: 事件1 事件2 使能因果 事件1 事件3 使能因果<｜end▁of▁sentence｜>


In [ ]:
from datasets import load_dataset
dataset=load_dataset("json", data_files="/content/train_data.jsonl")
print(dataset)
print("数据集的字段：", dataset.column_names)

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'event', 'relation'],
        num_rows: 378
    })
})
数据集的字段： {'train': ['text', 'event', 'relation']}


In [ ]:
'''
# 获取结束符，必须添加 EOS_TOKEN
EOS_TOKEN = tokenizer.eos_token

# 定义格式化函数，生成符合模板要求的 "text" 字段
def formatting_prompts_func(examples):
    new_texts = []
    # 根据数据集实际字段名称调整这里的字段
    # 这里假设数据集中包含 "instruction" 和 "output" 两个字段
    for instruction, output in zip(examples["prompt"], examples["completion"]):
        formatted_text = f"问题: {instruction}\n回答: {output}"
        new_texts.append(formatted_text)
    return {"text": new_texts}

# 对数据集应用格式化函数，生成符合模板要求的文本（即 "text" 字段）
dataset = dataset.map(formatting_prompts_func, batched=True)
'''
train_dataset = dataset['train']
# 查看第一个生成的文本
# print(dataset["text"][0])
print(train_dataset)

Dataset({
    features: ['text', 'event', 'relation'],
    num_rows: 378
})


In [ ]:
print(train_dataset[0]["relation"])

['(在; 青蛙; 瓶子里; 无; 无) (跑; 青蛙; 无; 无; 无) 使能-因果', '(睡; 他; 觉; 无; 无) (睡醒; 他; 无; 无; 无) 使能-因果', '(睡; 他; 觉; 无; 无) (跑; 青蛙; 无; 无; 无) 使能-因果', '(睡醒; 他; 无; 无; 无) (生气; 他; 无; 无; 无) 心理-因果', '(跑; 青蛙; 无; 无; 无) (生气; 他; 无; 无; 无) 心理-因果', '(跑; 青蛙; 无; 无; 无) (找; 他; 青蛙; 无; 无) 心理-因果', '(找; 他; 青蛙; 无; 无) (叫; 他; 青蛙; 无; 无) 使能-因果', '(找; 他; 青蛙; 无; 无) (想; 他; 青蛙在哪里; 无; 无) 使能-因果', '(找; 他; 青蛙; 无; 无) (打碎; 狗; 瓶子; 无; 无) 使能-因果', '(打碎; 狗; 瓶子; 无; 无) (生气; 他; 无; 无; 无) 心理-因果', '(找; 他; 青蛙; 无; 无) (叫; 他; 青蛙; 无; 无) 动机-因果', '(叫; 他; 青蛙; 无; 无) (没; 这个树林里; 青蛙; 无; 无) 使能-因果', '(没; 这个树林里; 青蛙; 无; 无) (到; 他; 上面; 无; 无) 使能-因果', '(找; 他; 青蛙; 无; 无) (到; 他; 上面; 无; 无) 动机-因果', '(到; 他; 上面; 无; 无) (摔; 无; 无; 无; 无) 使能-因果', '(跑; 小狗; 无; 无; 无) (摔; 无; 无; 无; 无) 并列', '(当成; 他; 梅花鹿; 无; 无) (摔; 梅花鹿; 他; 无; 无) 使能-因果', '(摔; 梅花鹿; 他; 无; 无) (摔; 无; 无; 无; 水里) 使能-因果', '(摔; 无; 无; 无; 水里) (坐; 他; 无; 无; 在泥坑上.) 使能-因果', '(坐; 他; 无; 无; 在泥坑上.) (找到; 他; 青蛙; 无; 无) 使能-因果', '(找; 他; 青蛙; 无; 无) (找到; 他; 青蛙; 无; 无) 动机-因果', '(找到; 他; 青蛙; 无; 无) (在; 青蛙; 这里; 无; 无) 使能-因果', '(在; 青蛙; 这里; 无; 无) (跳过来; 无; 无; 

In [ ]:
train_prompt_style = """###指令: 从儿童语言文本中提取叙事事件和事件关系。文本以<sen>分隔。
**事件定义:** (谓语；主语；宾语；时间状语；地点状语)  [缺失信息用“无”，多主语逗号分隔]
**事件关系:** [并列、动机因果、心理因果、物理因果、使能因果]
**输出格式:**
* 事件: (谓语；主语；宾语；时间状语；地点状语)  [多个事件用空格分隔]
* 关系: 事件1 事件2 关系类型  [不同事件关系用逗号分隔]
严格按格式输出。

###问题: {}
###回答: {}"""

EOS_TOKEN = tokenizer.eos_token

In [ ]:
def formatting_prompts_func(examples):  # Takes a batch of dataset examples as input
    inputs = examples["text"]       # Extracts the medical question from the dataset

    output1 = examples["event"]      # Extracts the final model-generated response (answer)
    output2 = examples["relation"]
    outputs = [f"事件：{o1}\n关系：{o2}" for o1, o2 in zip(output1, output2)]

    texts = []  # Initializes an empty list to store the formatted prompts

    # Iterate over the dataset, formatting each question, reasoning step, and response
    for input, output in zip(inputs, outputs):
        text = train_prompt_style.format(input, output) + EOS_TOKEN  # Insert values into prompt template & append EOS token
        texts.append(text)  # Add the formatted text to the list

    return {
        "text": texts,  # Return the newly formatted dataset with a "text" column containing structured prompts
    }

In [ ]:
dataset_finetune = train_dataset.map(formatting_prompts_func, batched = True)
dataset_finetune["text"][2]
# print(dataset_finetune)

Map:   0%|          | 0/378 [00:00<?, ? examples/s]

"###指令: 从儿童语言文本中提取叙事事件和事件关系。文本以<sen>分隔。\n**事件定义:** (谓语；主语；宾语；时间状语；地点状语)  [缺失信息用“无”，多主语逗号分隔]\n**事件关系:** [并列、动机因果、心理因果、物理因果、使能因果]\n**输出格式:**\n* 事件: (谓语；主语；宾语；时间状语；地点状语)  [多个事件用空格分隔]\n* 关系: 事件1 事件2 关系类型  [不同事件关系用逗号分隔]\n严格按格式输出。\n\n###问题: 这个狗在吃青蛙.<sen>这个也在看小狗吃青蛙.<sen>这个青蛙一个腿在里面.<sen>一个腿在外面.<sen>一个人腿在被子里面.<sen>在上面睡觉.<sen>这个人在抱着小狗睡觉.<sen>这个小狗和青蛙没有在被子里面.<sen>小狗没有在外面.<sen>这个人没有穿衣服,没有穿鞋,也没有穿裤子.<sen>有个人在家看一个动物,看到一个燕子.<sen>这个人在里面看小狗都掉下去了.<sen>这个小狗抱着人.<sen>这个家里面就没有人了.<sen>这个人在看叶子.<sen>这个树上有人.<sen>这个树上没有人.<sen>这个有毛毛虫的里面有石头.<sen>那这个树的上面有石头.<sen>在喊动物.<sen>对吧?<sen>这个树上面也在喊,青蛙.<sen>这个狗在摇石头.<sen>对吧?<sen>这个狗在爬树.<sen>这个人在爬树.<sen>这个人是摔了跤.<sen>这个没有.<sen>这个是人.<sen>这个是人和狗.<sen>这个人在爬雪山.<sen>这个人在喊动物飞来.<sen>对吧?<sen>这个人从雪山摔了一跤.<sen>那这个人爬着小狗掉到水里了.<sen>这个小看到一个人掉到水里去了.<sen>这个小朋友掉到水里.<sen>就头上有个小狗.<sen>坐在水里面.<sen>这个人在喊狗,快起来.<sen>这个人就爬上去了.<sen>这个人在上面,朝下穿到洞里面,然后再钻出来了.<sen>这个人还没有钻洞.<sen>还要爬下来,走着再进去,然后再钻出来了.<sen>这个人已经钻出来了.<sen>这么多的青蛙在看水.<sen>有荷叶.\n###回答: 事件：['(吃; 狗; 青蛙; 无; 无)', '(看; 这个; 小狗吃青蛙; 无; 无)', '(睡觉; 人; 无; 无; 无)', 

In [ ]:
dataset_dev=load_dataset("json", data_files="/content/dev_data.jsonl")
print(dataset_dev)

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'event', 'relation'],
        num_rows: 108
    })
})


In [ ]:
dev_dataset=dataset_dev['train']
print(dev_dataset[0])

{'text': '小狗小青蛙小朋友就要准备睡觉了.<sen>于是小青蛙没睡觉.<sen>他们两个睡觉了.<sen>然后他们醒了.<sen>咦?<sen>小青蛙不见了.7<sen>他们在找小青蛙.<sen>到处找啊找,找啊找.<sen>于是他在找啊.<sen>他掉下去了.<sen>小狗接住了他.<sen>然后他们在找小青蛙.<sen>地洞里是不是有小青蛙呢?不是小青蛙.<sen>他们在树洞里找.<sen>又不是小青蛙.<sen>他们在山上找.和小驯鹿就一起找了.掉下去了.<sen>他们在河里了.<sen>他掉进河底也要开始找小青蛙.<sen>嘘.<sen>然后他找啊找.忽然找到了两只小青蛙.<sen>还有几只小青蛙.<sen>然后他找到了小青蛙.<sen>他就高兴的笑了.', 'event': ['(睡觉; 他们两个; 无; 无; 无)', '(醒; 他们; 无; 无; 无)', '(不见; 小青蛙; 无; 无; 无)', '(找; 他们; 小青蛙; 无; 无)', '(掉下; 他; 无; 无; 无)', '(接住; 小狗; 他; 无; 无)', '(找; 他们; 小青蛙; 无; 无)', '(不是; 无; 小青蛙; 无; 地洞里)', '(找; 他们; 无; 无; 在树洞里)', '(找; 他们; 无; 无; 在山上)', '(掉下; 他们; 无; 无; 无)', '(在; 他们; 河里; 无; 无)', '(找; 他; 小青蛙; 无; 河底)', '(找到; 他; 两只小青蛙; 无; 无)', '(有; 无; 几只小青蛙; 无; 无)', '(找到; 他; 小青蛙; 无; 无)', '(笑; 他; 无; 无; 无)'], 'relation': ['(睡觉; 他们两个; 无; 无; 无) (醒; 他们; 无; 无; 无) 使能-因果', '(睡觉; 他们两个; 无; 无; 无) (不见; 小青蛙; 无; 无; 无) 使能-因果', '(不见; 小青蛙; 无; 无; 无) (找; 他们; 小青蛙; 无; 无) 心理-因果', '(不见; 小青蛙; 无; 无; 无) (找; 他们; 小青蛙; 无; 无) 心理-因果', '(掉下; 他; 无; 无; 无) (接住; 小狗; 他; 无; 无) 使能-因果', '(找; 他们; 小青蛙; 无; 无) (不是; 无; 小青

In [ ]:
dataset_finetune_dev = dev_dataset.map(formatting_prompts_func, batched = True)
dataset_finetune_dev["text"][2]

Map:   0%|          | 0/108 [00:00<?, ? examples/s]

"###指令: 从儿童语言文本中提取叙事事件和事件关系。文本以<sen>分隔。\n**事件定义:** (谓语；主语；宾语；时间状语；地点状语)  [缺失信息用“无”，多主语逗号分隔]\n**事件关系:** [并列、动机因果、心理因果、物理因果、使能因果]\n**输出格式:**\n* 事件: (谓语；主语；宾语；时间状语；地点状语)  [多个事件用空格分隔]\n* 关系: 事件1 事件2 关系类型  [不同事件关系用逗号分隔]\n严格按格式输出。\n\n###问题: 小狗要吃掉小青蛙.<sen>小朋友和小狗睡着了.<sen>小青蛙静悄悄的从瓶子里出来了.<sen>小朋友他看到小青蛙从瓶子里不见了.<sen>然后小青蛙就一下靴子里有青蛙吗.<sen>小狗从里面闻了闻味道.<sen>小狗掉下来窗台.<sen>小朋友接住小狗.<sen>小狗看看树上有没有小青蛙.<sen>树上有蜜蜂们.<sen>看看地洞里有没有小青蛙.<sen>钻出来小鼹鼠.<sen>不是小青蛙.<sen>一群蜜蜂追着小狗狗.<sen>小朋友又上树.<sen>看看树洞里有青蛙吗.<sen>没有.<sen>猫头鹰飞下来.<sen>把小朋友吓了一大跳.<sen>蜜蜂还在那儿追狗.<sen>小朋友又上去了.<sen>小男孩上去.<sen>猫头鹰还在吓他.<sen>猫头鹰睡觉了.<sen>小朋友在抓住两根树枝.<sen>结果鹿把他背了.<sen>鹿就背着他跑跑跑.<sen>跑下悬崖.<sen>他和小狗都掉下了悬崖.<sen>掉到了水里.<sen>他又出来了.<sen>然后他们爬上了木桩看看.<sen>木桩后面有小青蛙和他的爸爸<sen>在一起生活.<sen>有更多的小青蛙.<sen>她托起了一只小青蛙说<sen>再见再见.\n###回答: 事件：['(要; 小狗; 吃掉小青蛙; 无; 无)', '(睡着; 小朋友，小狗; 无; 无; 无)', '(出来; 小青蛙; 无; 无; 从瓶子里)', '(看到; 小朋友; 小青蛙; 无; 从瓶子里)', '(闻了闻; 小狗; 味道; 无; 无)', '(掉下; 小狗; 无; 无; 窗台)', '(接住; 小朋友; 小狗; 无; 无)', '(看看; 小狗; 树上有没有小青蛙; 无; 无)', '(有; 树上; 蜜蜂们; 无; 无)', '(看看; 无; 有没有小青蛙; 无;

In [ ]:
# 假设 dataset_finetune 是你的数据集
max_token_length_data = 0  # 用于记录最长的 token 数量

# 遍历数据集中的每条文本
for text in dataset_finetune["text"]:
    # 使用 tokenizer 对文本进行编码
    tokens = tokenizer(text, return_tensors="pt", truncation=False)["input_ids"]
    # 获取 token 数量
    token_length = tokens.shape[1]
    # 更新最大 token 数量
    if token_length > max_token_length_data:
        max_token_length_data = token_length

print(f"数据集中最长的 token 数量是: {max_token_length_data}")

数据集中最长的 token 数量是: 5872


In [ ]:
# 假设 dataset_finetune 是你的数据集
total_token_length = 0  # 用于累加所有文本的 token 数量
num_texts = len(dataset_finetune["text"])  # 数据集中的文本数量

# 遍历数据集中的每条文本
for text in dataset_finetune["text"]:
    # 使用 tokenizer 对文本进行编码
    tokens = tokenizer(text, return_tensors="pt", truncation=False)["input_ids"]
    # 获取 token 数量
    token_length = tokens.shape[1]
    # 累加 token 数量
    total_token_length += token_length

# 计算平均 token 数量
average_token_length = total_token_length / num_texts

print(f"数据集中每条文本的平均 token 数量是: {average_token_length}")

数据集中每条文本的平均 token 数量是: 2305.2751322751324


In [ ]:
count = 0
for text in dataset_finetune["text"]:
    # 使用 tokenizer 对文本进行编码
    tokens = tokenizer(text, return_tensors="pt", truncation=False)["input_ids"]
    # 获取 token 数量
    token_length = tokens.shape[1]
    if token_length > 4096:
        count+=1
print(count)

14


In [ ]:
model_lora = FastLanguageModel.get_peft_model(
    model=model,  # 待微调的模型
    r=8,  # LoRA 分解的秩，保持为 8，适合大型模型和大数据集
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        # 仅对注意力头的投影层应用 LoRA，符合 Qwen 模型架构
    ],
    lora_alpha=8,  # 调整为 8，与 r 匹配，结合 RSLoRA 稳定训练
    lora_dropout=0.1,  # 保持 0.1，防止过拟合，适合大数据集
    bias="none",  # 不修改偏置项，保持默认设置
    use_gradient_checkpointing=True,  # 启用梯度检查点，节省显存，适合 32B 模型
    random_state=527,  # 固定随机种子，确保训练可复现
    use_rslora=True,  # 启用 RSLoRA，提升训练稳定性
    loftq_config=None,  # 保持示例配置，可根据需求调整
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.1.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2025.3.9 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported, FastLanguageModel

In [ ]:
trainer = SFTTrainer(
    model=model_lora,  # The model to be fine-tuned
    tokenizer=tokenizer,  # Tokenizer to process text inputs
    train_dataset=dataset_finetune,  # Dataset used for training
    eval_dataset=dataset_finetune_dev,  # Dataset used for evaluation (optional)
    dataset_text_field="text",  # Specifies which field in the dataset contains training text
    max_seq_length=max_seq_length,  # Defines the maximum sequence length for inputs
    dataset_num_proc=2,  # Uses 2 CPU threads to speed up data preprocessing

    # Define training arguments
    args=TrainingArguments(
        per_device_train_batch_size=4,  # Number of examples processed per device (GPU) at a time
        gradient_accumulation_steps=2,  # Accumulate gradients over 4 steps before updating weights
        num_train_epochs=10, # Full fine-tuning run
        warmup_steps=5,  # Gradually increases learning rate for the first 5 steps
        # max_steps=60,  # Limits training to 60 steps (useful for debugging; increase for full fine-tuning)
        learning_rate=2e-4,  # Learning rate for weight updates (tuned for LoRA fine-tuning)
        fp16=not is_bfloat16_supported(),  # Use FP16 (if BF16 is not supported) to speed up training
        bf16=is_bfloat16_supported(),  # Use BF16 if supported (better numerical stability on newer GPUs)
        logging_steps=10,  # Logs training progress every 10 steps
        optim="adamw_8bit",  # Uses memory-efficient AdamW optimizer in 8-bit mode
        weight_decay=0.01,  # Regularization to prevent overfitting
        lr_scheduler_type="linear",  # Uses a linear learning rate schedule
        seed=527,  # Sets a fixed seed for reproducibility
        output_dir="/content/outputs",  # Directory where fine-tuned model checkpoints will be saved

        eval_strategy="steps",      # 启用按步骤评估
        eval_steps=10,             # 每 10 步评估一次
        per_device_eval_batch_size=4,      # 验证批次大小
    ),
)

Tokenizing to ["text"] (num_proc=2):   0%|          | 0/378 [00:00<?, ? examples/s]

Tokenizing to ["text"] (num_proc=2):   0%|          | 0/108 [00:00<?, ? examples/s]

In [ ]:
wnb_token=userdata.get('wandb_token')

In [ ]:
import wandb

In [ ]:
# Login to WnB
wandb.login(key=wnb_token) # import wandb
run = wandb.init(
    project='test0312',
    entity='FeSCN',
    job_type="training",
    settings=wandb.Settings(init_timeout=120),
    anonymous="allow"
)

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: yuxuan0612 (FeSCN) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 378 | Num Epochs = 10 | Total steps = 470
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 5,046,272/5,348,005,376 (0.09% trained)
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
10,1.016600,1.038132
20,0.871700,0.874976
30,0.792000,0.750653
40,0.663000,0.703802
50,0.643500,0.680397
60,0.624100,0.663547
70,0.582900,0.649265
80,0.587600,0.636413
90,0.567900,0.627019
100,0.535900,0.620744


Unsloth: Not an error, but Qwen2ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


In [ ]:
from unsloth import unsloth_train
trainer_stats = unsloth_train(trainer)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 378 | Num Epochs = 10 | Total steps = 470
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 5,046,272/5,348,005,376 (0.09% trained)


Step,Training Loss,Validation Loss
10,0.852500,0.828891
20,0.743200,0.690256
30,0.705700,0.642417
40,0.633800,0.617268
50,0.618500,0.598707
60,0.603100,0.584352
70,0.567400,0.572935
80,0.574500,0.564540
90,0.557500,0.555839
100,0.526700,0.550056


Unsloth: Not an error, but Qwen2ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


In [ ]:
question = """成千上万的蜜蜂都飞过来了<sen>然后呢因为蜜蜂追着他<sen>然后呢蜂巢掉下来了"""
print(train_prompt_style.format(question, ""))

###指令: 从儿童语言文本中提取叙事事件和事件关系。文本以<sen>分隔。
**事件定义:** (谓语；主语；宾语；时间状语；地点状语)  [缺失信息用“无”，多主语逗号分隔]
**事件关系:** [并列、动机因果、心理因果、物理因果、使能因果]
**输出格式:**
* 事件: (谓语；主语；宾语；时间状语；地点状语)  [多个事件用空格分隔]
* 关系: 事件1 事件2 关系类型  [不同事件关系用逗号分隔]
严格按格式输出。

###问题: 成千上万的蜜蜂都飞过来了<sen>然后呢因为蜜蜂追着他<sen>然后呢蜂巢掉下来了
###回答: 


In [ ]:
question = """小男孩在家里养了一只青蛙.<sen>狗也在看这只青蛙.<sen>然后呢小男孩睡着了.<sen>然后青蛙从这瓶口里跑出来了.<sen>然后他看下啊.<sen>青蛙怎么没啦.<sen>然后他找帽子里.<sen>帽子里没有.<sen>外面也没有.<sen>然后呢狗套着罐子出去了.<sen>然后小男孩抱住了这只狗.<sen>然后他正在喊青蛙.<sen>然后呢狗站在蜂巢底下.<sen>然后蜜蜂飞出来了.<sen>然后小男孩往洞里看.<sen>洞里出来的而不是青蛙.<sen>是小老鼠.<sen>然后呢蜂巢掉下来了.<sen>成千上万的蜜蜂都飞过来了.<sen>这个小男孩嗯在看树洞里有没有青蛙.<sen>然后小男孩在地上了.<sen>然后呢因为蜜蜂追着他.<sen>然后所以狗就跑了.<sen>然后猫头鹰也追着小男孩.<sen>小男孩跑了.<sen>然后小男孩站在石头上喊青蛙.<sen>然后呢小男孩在小鹿上找.<sen>然后呢狗也跑了.<sen>然后呢小男孩掉下去了.<sen>然后呢小男孩掉到了水里.<sen>然后呢小男孩说嘘.<sen>然后狗狗翻过了这个木头.<sen>它发现了两只青蛙.<sen>然后青蛙又更多了.<sen>然后他就看着这些青蛙.<sen>然后他就很高兴.<sen>然后呢小男孩就很高兴."""

# Load the inference model using FastLanguageModel (Unsloth optimizes for speed)
FastLanguageModel.for_inference(model_lora)  # Unsloth has 2x faster inference!

# Tokenize the input question with a specific prompt format and move it to the GPU
inputs = tokenizer([train_prompt_style.format(question, "")], return_tensors="pt").to("cuda")

# Generate a response using LoRA fine-tuned model with specific parameters
outputs = model_lora.generate(
    input_ids=inputs.input_ids,          # Tokenized input IDs
    attention_mask=inputs.attention_mask, # Attention mask for padding handling
    max_new_tokens=1200,                  # Maximum length for generated response
    use_cache=True,                        # Enable cache for efficient generation
)

# Decode the generated response from tokenized format to readable text
response = tokenizer.batch_decode(outputs)

# Extract and print only the model's response part after "### Response:"
print(response[0].split("###回答:")[1])

 事件：['(养; 小男孩; 青蛙; 无; 在家里)', '(看; 狗; 青蛙; 无; 无)', '(睡着; 小男孩; 无; 无; 无)', '(跑出; 青蛙; 无; 无; 从这瓶口里)', '(看; 他; 无; 无; 无)', '(没有; 帽子里; 无; 无; 无)', '(找; 他; 无; 无; 帽子里)', '(套着; 罐子; 狗; 无; 无)', '(出去; 罐子; 狗; 无; 无)', '(抱; 小男孩; 狗; 无; 无)', '(喊; 他; 青蛙; 无; 无)', '(站在; 狗; 蜂巢底下; 无; 无)', '(飞; 蜜蜂; 无; 无; 无)', '(看; 小男孩; 洞里; 无; 无)', '(出来; 小老鼠; 无; 无; 无)', '(掉; 蜂巢; 无; 无; 无)', '(追; 蜜蜂; 他; 无; 无)', '(追; 猫头鹰; 小男孩; 无; 无)', '(跑; 狗; 无; 无; 无)', '(追; 猫头鹰; 小男孩; 无; 无)', '(站; 小男孩; 无; 无; 在石头上)', '(找; 小男孩; 无; 无; 在小鹿上)', '(跑; 狗; 无; 无; 无)', '(掉; 小男孩; 无; 无; 无)', '(到; 小男孩; 水里; 无; 无)', '(翻过; 狗; 木头; 无; 无)', '(发现; 它; 两只青蛙; 无; 无)', '(看; 他; 青蛙; 无; 无)', '(高兴; 小男孩; 无; 无; 无)']
关系：['(养; 小男孩; 青蛙; 无; 在家里) (看; 狗; 青蛙; 无; 无) 使能-因果', '(睡着; 小男孩; 无; 无; 无) (跑出; 青蛙; 无; 无; 从这瓶口里) 使能-因果', '(跑出; 青蛙; 无; 无; 从这瓶口里) (看; 他; 无; 无; 无) 使能-因果', '(跑出; 青蛙; 无; 无; 从这瓶口里) (没有; 帽子里; 无; 无; 无) 使能-因果', '(没有; 帽子里; 无; 无; 无) (找; 他; 无; 无; 帽子里) 心理-因果', '(找; 他; 无; 无; 帽子里) (套着; 罐子; 狗; 无; 无) 动机-因果', '(套着; 罐子; 狗; 无; 无) (出去; 罐子; 狗; 无; 无) 使能-因果', '(出去; 罐子; 狗; 无; 无) (抱; 小男孩; 狗; 无; 无) 

In [ ]:
question = """小男孩在家里养了一只青蛙.<sen>狗也在看这只青蛙.<sen>然后呢小男孩睡着了.<sen>然后青蛙从这瓶口里跑出来了.<sen>然后他看下啊.<sen>青蛙怎么没啦.<sen>然后他找帽子里.<sen>帽子里没有.<sen>外面也没有.<sen>然后呢狗套着罐子出去了.<sen>然后小男孩抱住了这只狗.<sen>然后他正在喊青蛙.<sen>然后呢狗站在蜂巢底下.<sen>然后蜜蜂飞出来了.<sen>然后小男孩往洞里看.<sen>洞里出来的而不是青蛙.<sen>是小老鼠.<sen>然后呢蜂巢掉下来了.<sen>成千上万的蜜蜂都飞过来了.<sen>这个小男孩嗯在看树洞里有没有青蛙.<sen>然后小男孩在地上了.<sen>然后呢因为蜜蜂追着他.<sen>然后所以狗就跑了.<sen>然后猫头鹰也追着小男孩.<sen>小男孩跑了.<sen>然后小男孩站在石头上喊青蛙.<sen>然后呢小男孩在小鹿上找.<sen>然后呢狗也跑了.<sen>然后呢小男孩掉下去了.<sen>然后呢小男孩掉到了水里.<sen>然后呢小男孩说嘘.<sen>然后狗狗翻过了这个木头.<sen>它发现了两只青蛙.<sen>然后青蛙又更多了.<sen>然后他就看着这些青蛙.<sen>然后他就很高兴.<sen>然后呢小男孩就很高兴."""

# Load the inference model using FastLanguageModel (Unsloth optimizes for speed)
FastLanguageModel.for_inference(model_lora)  # Unsloth has 2x faster inference!

# Tokenize the input question with a specific prompt format and move it to the GPU
inputs = tokenizer([train_prompt_style.format(question, "")], return_tensors="pt").to("cuda")

# Generate a response using LoRA fine-tuned model with specific parameters
outputs = model_lora.generate(
    input_ids=inputs.input_ids,          # Tokenized input IDs
    attention_mask=inputs.attention_mask, # Attention mask for padding handling
    max_new_tokens=1200,                  # Maximum length for generated response
    use_cache=True,                        # Enable cache for efficient generation
)

# Decode the generated response from tokenized format to readable text
response = tokenizer.batch_decode(outputs)

# Extract and print only the model's response part after "### Response:"
print(response[0].split("###回答:")[1])

 事件：['(养; 小男孩; 青蛙; 无; 无)', '(看; 狗; 青蛙; 无; 无)', '(睡着; 小男孩; 无; 无; 无)', '(跑; 青蛙; 无; 无; 无)', '(看; 他; 无; 无; 无)', '(找; 他; 帽子里; 无; 无)', '(没有; 帽子里; 无; 无; 无)', '(套; 狗; 罐子; 无; 无)', '(出去; 狗; 无; 无; 无)', '(抱住; 小男孩; 狗; 无; 无)', '(喊; 他; 青蛙; 无; 无)', '(站在; 狗; 无; 无; 蜂巢底下)', '(飞; 蜜蜂; 无; 无; 无)', '(看; 小男孩; 洞里; 无; 无)', '(出来; 小老鼠; 无; 无; 无)', '(掉; 蜂巢; 无; 无; 无)', '(飞; 蜜蜂; 无; 无; 无)', '(追; 蜜蜂; 他; 无; 无)', '(跑; 狗; 无; 无; 无)', '(追; 猫头鹰; 小男孩; 无; 无)', '(跑; 小男孩; 无; 无; 无)', '(站; 小男孩; 石头; 无; 无)', '(找; 小男孩; 无; 无; 小鹿上)', '(跑; 狗; 无; 无; 无)', '(掉; 小男孩; 无; 无; 无)', '(到; 小男孩; 水里; 无; 无)', '(说; 小男孩; 嘘; 无; 无)', '(翻; 狗狗; 木头; 无; 无)', '(发现; 它; 两只青蛙; 无; 无)', '(看; 他; 青蛙; 无; 无)', '(高兴; 小男孩; 无; 无; 无)']
关系：['(养; 小男孩; 青蛙; 无; 无) (看; 狗; 青蛙; 无; 无) 使能-因果', '(睡着; 小男孩; 无; 无; 无) (跑; 青蛙; 无; 无; 无) 使能-因果', '(跑; 青蛙; 无; 无; 无) (看; 他; 无; 无; 无) 使能-因果', '(看; 他; 无; 无; 无) (找; 他; 帽子里; 无; 无) 动机-因果', '(找; 他; 帽子里; 无; 无) (没有; 帽子里; 无; 无; 无) 使能-因果', '(看; 他; 无; 无; 无) (套; 狗; 罐子; 无; 无) 并列', '(套; 狗; 罐子; 无; 无) (出去; 狗; 无; 无; 无) 使能-因果', '(出去; 狗; 无; 无; 无) (抱住; 小男孩; 狗; 无; 无)

In [ ]:
wandb.finish()

eval/loss,█▅▄▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
eval/runtime,█▂▁▁▁▂▂▂▂▂▂▂▂▂▁▁▂▂▂▁▁▂▁▂▂▂▂▂▂▂▂▃▂▂▂▂▂▂▂▂
eval/samples_per_second,▁████▇▇▇▇▇▇▇▇▇██▇▇▆██▇█▇▇▇▆▇▆▇▇▆▆▇▇▇▇▇▇▇
eval/steps_per_second,▁▇▇█▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇
train/epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▆▆▆▆▆▇▇▇▇▇▇██
train/global_step,▁▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇█
train/grad_norm,▁▁▃▂▁▂▃▂▃▂▂▂▄▂▂▄▂▃▃▄█▃▃▅▄▄▄▄▄▃▅▄▄▅▆▅▅▄▅▃
train/learning_rate,█████▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▁▁▁
train/loss,█▆▅▄▄▃▃▃▃▂▂▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
eval/loss,0.50634
eval/runtime,79.6992


In [ ]:
new_model_online = "DeepSeek-R1-Distill-Qwen-7B-Children_Narrative_Extraction-Fine-tune_version2"
new_model_local = "/content/model/0312"


model_lora.save_pretrained("/content/model/lora") # Local saving
tokenizer.save_pretrained("/content/model/lora")

('/content/model/lora/tokenizer_config.json',
 '/content/model/lora/special_tokens_map.json',
 '/content/model/lora/tokenizer.json')

In [ ]:
model_lora.push_to_hub(new_model_online) # Online saving
tokenizer.push_to_hub(new_model_online) # Online saving

README.md:   0%|          | 0.00/624 [00:00<?, ?B/s]

  0%|          | 0/1 [00:00<?, ?it/s]

adapter_model.safetensors:   0%|          | 0.00/20.2M [00:00<?, ?B/s]

Saved model to https://huggingface.co/DeepSeek-R1-Distill-Qwen-7B-Children_Narrative_Extraction-Fine-tune


  0%|          | 0/1 [00:00<?, ?it/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [ ]:
model_lora.save_pretrained_merged(new_model_local, tokenizer, save_method = "merged_16bit",)
model_lora.push_to_hub_merged(new_model_online, tokenizer, save_method = "merged_16bit")

Unsloth: Kaggle/Colab has limited disk space. We need to delete the downloaded
model which will save 4-16GB of disk space, allowing you to save on Kaggle/Colab.
Unsloth: Will remove a cached repo with size 8.5G


Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 50.2 out of 83.48 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


100%|██████████| 28/28 [00:00<00:00, 149.23it/s]

Unsloth: Saving tokenizer...

 Done.
Done.
Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 50.05 out of 83.48 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


...

model-00002-of-00004.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.33G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Done.
Saved merged model to https://huggingface.co/None/DeepSeek-R1-Distill-Qwen-7B-Children_Narrative_Extraction-Fine-tune_version2


# **推理！！！测试结果保存到本地**

In [ ]:
from tqdm import tqdm
import os
import json

In [ ]:
# 确保输出目录存在
output_dir = "/content/generated_responses"
os.makedirs(output_dir, exist_ok=True)

# 读取 jsonl 文件的第一条数据
with open("eval_data.jsonl", "r") as f:
    first_line = f.readline().strip()  # 读取第一行并去掉首尾空格

# 解析 JSON 获取文本
data = json.loads(first_line)
question = data["text"]

# Tokenize 并移动到 GPU
inputs = tokenizer([train_prompt_style.format(question, "")], return_tensors="pt").to("cuda")

# 生成模型输出
outputs = model_lora.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=2048,
    use_cache=True,
)

# 解码输出
response = tokenizer.batch_decode(outputs)

# 提取模型回答部分
model_response = response[0].split("###回答:")[1]

# 保存到单个文件
output_path = os.path.join(output_dir, "response_test.txt")
with open(output_path, "w", encoding="utf-8") as out_file:
    out_file.write(model_response)

print(f"测试数据已处理，结果保存在 {output_path}")

测试数据已处理，结果保存在 /content/generated_responses/response_test.txt


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
drive_output_dir="/content/drive/MyDrive/results_0312"

In [ ]:
# Load the inference model using FastLanguageModel (Unsloth optimizes for speed)
FastLanguageModel.for_inference(model_lora)  # Unsloth has 2x faster inference!

# Open the test set (jsonl format)
with open("eval_data.jsonl", "r") as f:
    lines = f.readlines()

# Iterate through the test data and process each entry
for idx, line in tqdm(enumerate(lines), desc="Processing Test Set"):
    # Parse the JSON line to get the text data
    data = json.loads(line)
    question = data["text"]

    # Tokenize the input question with a specific prompt format and move it to the GPU
    inputs = tokenizer([train_prompt_style.format(question, "")], return_tensors="pt").to("cuda")

    # Generate a response using LoRA fine-tuned model with specific parameters
    outputs = model_lora.generate(
        input_ids=inputs.input_ids,          # Tokenized input IDs
        attention_mask=inputs.attention_mask, # Attention mask for padding handling
        max_new_tokens=4096,                  # Maximum length for generated response
        use_cache=True,                        # Enable cache for efficient generation
    )

    # Decode the generated response from tokenized format to readable text
    response = tokenizer.batch_decode(outputs)

    # Extract the model's response part after "### Response:"
    model_response = response[0].split("###回答:")[1]

    output_file_path = os.path.join(drive_output_dir, f"result_{idx + 1}.txt")
    with open(output_file_path, "w", encoding="utf-8") as out_file:
        out_file.write(model_response)


Processing Test Set: 55it [1:37:11, 106.02s/it]
